# 2018–2020 韩国关税署 10 月 1–20 日原稿索引

三个冻结官网日期查询，NAS 原始页备份和来源标题/日期/机构匹配；不读 SOXX、拟合或跑账户。

In [1]:
import hashlib,json,subprocess,tempfile
from datetime import datetime,timezone,date
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot
root=Path.cwd();card=root/'research/kcs_early20_2018_2020_oct_official_index_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
raw=root/'data/snapshots/kcs_early20_2018_2020_oct_official_index_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];failure=None
for spec in plan['exact_requests']:
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    body=raw/spec['name'];headers=raw/(spec['name']+'.headers');started=datetime.now(timezone.utc).isoformat();t=plan['transport']
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(t['connect_timeout_seconds']),'--max-time',str(t['total_timeout_seconds_each']),'--max-filesize',str(t['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',t['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
    try:
        run=subprocess.run(cmd,capture_output=True,text=True,timeout=35);parts=run.stdout.split(' ',1);rec={'year':spec['year'],'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':int(parts[0]) if parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'transport_error':run.stderr[:240]}
    except Exception as exc:rec={'year':spec['year'],'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
    rec.update(body_sha256=sha(body) if body.exists() else None,body_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
    receipts.append(rec);(raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
    if rec.get('curl_exit_code')!=0 or rec.get('http_status')!=200 or not body.exists() or body.stat().st_size>t['max_response_bytes_each'] or b'pressReleaseList' not in body.read_bytes():failure='transport_or_index_identity:'+str(spec['year']);break
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'receipts':receipts,'failure':failure,'historical_first_public_bytes_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'Korea government KCS 2018-2020 October press release list'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='kcs-2018-2020-index-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
except Exception as exc:failure=(failure+';' if failure else '')+'NAS:'+type(exc).__name__+':'+str(exc)[:180];binding=None
pages=[]
if not failure:
    for spec in plan['exact_requests']:
        soup=BeautifulSoup((raw/spec['name']).read_bytes(),'html.parser');items=[]
        for link in soup.select('div.list_type li a[href*="pressReleaseView"]'):
            title=link.select_one('strong');title=title.get_text(' ',strip=True) if title else link.get_text(' ',strip=True)
            spans=[z.get_text(' ',strip=True) for z in link.select('.source span')]
            row={'title':title,'display_date':spans[0] if spans else None,'agency':spans[1] if len(spans)>1 else None,'url':urljoin('https://www.korea.kr',link.get('href',''))}
            items.append(row)
        matches=[z for z in items if '10월' in z['title'] and '20일' in z['title'] and '수출입' in z['title'] and z['agency']=='관세청']
        pages.append({'year':spec['year'],'listed_items':len(items),'matched_KCS_Oct_early20':matches,'pagination_numbers':[z.get_text(strip=True) for z in soup.select('div.paging span.num')][:12]})
state.update(status='completed' if not failure else 'stopped',failure=failure);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'official_KCS_2018_2020_Oct_early20_index_leads_mapped_source_only' if not failure else 'needs_official_KCS_index_or_NAS','failure':failure,'GET_attempts':state['GET_attempts'],'pages':pages,'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':bool(binding),'historical_first_public_bytes_qualified':False,'full_monthly_training_panel_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'strategy_forward_days':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps(result,ensure_ascii=False))


{"decision": "official_KCS_2018_2020_Oct_early20_index_leads_mapped_source_only", "failure": null, "GET_attempts": 3, "pages": [{"year": 2018, "listed_items": 3, "matched_KCS_Oct_early20": [{"title": "2018년 10월 1일 ∼ 10월 20일 수출입 현황", "display_date": "2018-10-22", "agency": "관세청", "url": "https://www.korea.kr/briefing/pressReleaseView.do?newsId=156299697&pageIndex=1&repCodeType=정부부처&repCode=B00003&startDate=2018-10-20&endDate=2018-10-25&srchWord=수출입&period="}], "pagination_numbers": ["1"]}, {"year": 2019, "listed_items": 1, "matched_KCS_Oct_early20": [{"title": "’19년 10월 1일 ∼ 10월 20일 수출입 현황", "display_date": "2019-10-21", "agency": "관세청", "url": "https://www.korea.kr/briefing/pressReleaseView.do?newsId=156356469&pageIndex=1&repCodeType=정부부처&repCode=B00003&startDate=2019-10-20&endDate=2019-10-25&srchWord=수출입&period="}], "pagination_numbers": ["1"]}, {"year": 2020, "listed_items": 3, "matched_KCS_Oct_early20": [{"title": "’20년 10월 1일 ∼ 10월 20일 수출입 현황", "display_date": "2020-10-21", "agency

# 2018–2020 年韩国关税署 10 月 1–20 日原稿索引

按事前固定的三个韩国政府政策门户日期查询，2018、2019、2020 年各返回一个标题、机构和日期均匹配的关税署 10 月 1–20 日进出口原稿入口，分页均只有一页：**2018-10-22** `newsId=156299697`、**2019-10-21** `newsId=156356469`、**2020-10-21** `newsId=156416374`。三份确切列表 HTML、响应头、接收时间和 SHA-256 已存 Yanxuan NAS 并远端核对、定向恢复。对应页面入口与匹配字段见 `result.json`。

与已经取得的 2021、2022 年政府原稿页一起，**2018–2022 五个 10 月发布入口均可定位**。本卡只证明当前官方目录能找到这些历史页面，尚未收取 2018–2020 的附件、提取半导体表格、核对版本或首次公开时刻；五个 10 月样本更不是完整月度训练序列。没有读取 SOXX 结果、拟合、账户或策略前瞻日。下一来源步骤应先冻结每页附件选择规则，收取原字节并逐年核对类别、单位和暂定修订语义，之后才谈 Qlib 信息门与完整 Stage1 0bp 账户。
